<a href="https://colab.research.google.com/github/AdaTatulescu/mlp_vs_cnn_cats_dogs_classification/blob/main/Cats_and_dogs_MLPCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P1. Classification on Cats vs Dogs dataset

Task: load the cats vs dogs dataset, write a simple neural network to perform classification over the two classes, test the network and plot the results for each class.

![Animals-10](https://storage.googleapis.com/kaggle-datasets-images/59760/115796/95649e825d9344084af2a24012c4d072/dataset-cover.jpg?t=2018-10-05-08-07-40)

## Task 0: Import libraries ✅

All the libraries in this section should be enough for the project. If you want to, you can import more libraries.

In [ ]:
# We need to first install PyTorch Lightning, since it's not included in Python
!pip install pytorch-lightning

In [ ]:
# Matplotlib for plotting, numpy for math operations
from matplotlib import pyplot as plt
import numpy as np

# PyTorch Lightning ⚡ to define our neural network
import torch
import torch.nn as nn
from torch.nn import functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer

# Dataloader and torchvision to define our input dataset
from torch.utils.data import random_split, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

# Torchmetrics to monitor metrics (classification accuracy)
from torchmetrics.functional.classification.accuracy import accuracy

## Task 1: Download/Import Dataset 📃
Link full dataset: https://www.kaggle.com/datasets/alinasir1596/catvsdog-small-dataset?resource=download

Link local version: https://drive.google.com/file/d/1KLwUcMDodGiioqODrczIQZpU7Jw4qgxi/view?usp=share_link

In [ ]:
import os
# Make sure to upload the data in your drive
# Importing our dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset_path = "/content/drive/MyDrive/..."


## Task 1: Define parameters and hyperparameters 📃


*   **Learning rate** - how fast does our network learn? (Warning, do not set too high)
*   **Batch size** - how many samples does the network "see" at every iteration (usually, the bigger the better)
*   **Dataset size** - how many samples are in the whole dataset? (train + validation)
*   **Train size** - how many samples are in the training dataset?
*   **Val size** - how many samples are in the validation dataset?
*   **Input width** - how wide (in pixels) is our input image? (must be square images)
*   **Input height** - how tall (in pixels) is our input image? (must be square images)
*   **Input channels** - how many channels does the input image have?
*   **Mean** - dataset mean
*   **Std** - dataset standard variance
*   **Hidden dim** - how many neurons are there in the hidden layer?
*   **Num classes** - how many classes (digits) do we want to classify in our dataset?





### Optimization Hyperparameters

In [ ]:
input_channels = 3
input_width = 64
input_height = 64
hidden_dim = 64
num_classes = 2
learning_rate = 1e-3
batch_size = 32



-> input_channels = 3 : images have 3 color channels: Red, Green, Blue (RGB images)

-> input_width = 64 :
input_height = 64 fixed sized input images for MLP, scaled to 64x64 pixels

-> hidden_dim = 64 : number of neurons in a hidden layer of the MLP

-> num_classes = 2 : dogs and cats; two classes of classification

-> learning_rate = 1e-3 : stable and fast while training networks like Adam

-> batch_size = 32 : balance between train and memory use when processing more images at the same time

### Data Augmentation

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]


train_transform = transforms.Compose([
    transforms.Resize((64, 64)),             # resizes each image to 64x64 pixels
    transforms.RandomHorizontalFlip(p=0.5),  # recomposes the same image into more images to reduce overfitting
    transforms.RandomRotation(15),           # photos are not perfectly straight, simulates variations
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # less sensitivity to illumination
    transforms.ToTensor(),                   # PyTorch tensor conversion
    transforms.Normalize(mean, std)          # standardizes image tensor with values from ImageNet that work for natural images
])

val_test_transform = transforms.Compose([
    transforms.Resize((64, 64)),            # resize each image to 64x64 pixels
    transforms.ToTensor(),                  # Pytorch tensor conversion
    transforms.Normalize(mean, std)         # standardizes image tensor with values from ImageNet that work for natural images
])


### Before and After Data Augmentation

Before

manual_transform = transforms.Compose

([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

-> the model memorizes the images instead of learning patterns

-> poor generalization

-> high chances of overfitting

After

->the model learns how the image is created, not the image itself

-> good generalization

-> good performance on new data

-> simulates new images from a single image



### Before and After Data Augmentation

In [ ]:
import random
from PIL import Image
from torchvision import transforms
target_class = 'dogs' # Change to 'cats' if preferred
source_dir = os.path.join(dataset_path, 'train', target_class)

# Random file
random_filename = random.choice(os.listdir(source_dir))
img_path = os.path.join(source_dir, random_filename)

# Augmentation Pipeline
demo_transform = transforms.Compose([
    transforms.Resize((128, 128)),            # Same resize as your model
    transforms.RandomHorizontalFlip(p=1.0),   # flip
    transforms.RandomRotation(degrees=15),    # rotation
    transforms.ColorJitter(brightness=0.2, contrast=0.2) # color jitter
])

# 3. Process the Images
original_img = Image.open(img_path).convert("RGB")
# We resize original just for side-by-side comparison consistency
original_resized = original_img.resize((128, 128))

# Apply the augmentations
augmented_img = demo_transform(original_img)

# 4. Plot for the Presentation
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Original
axes[0].imshow(original_resized)
axes[0].set_title("Image before augmentation", fontsize=12)
axes[0].axis('off')

# Augmented
axes[1].imshow(augmented_img)
axes[1].set_title("Image after augmentation", fontsize=12, color='blue')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Task 2: Define the DataModule 📃

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
import pytorch_lightning as pl

# DataModule for uploading images ( train, val, test)
class MyCustomDataModule(pl.LightningDataModule):
    def __init__(self, data_dir):
        super().__init__()
        self.data_dir = data_dir
        # image transformation
        self.train_transform = train_transform        # used in training
        self.val_test_transform = val_test_transform  # used in validation and testing

    def prepare_data(self):
        pass

    def setup(self, stage=None):
      # create a different dataset depending on each stage
        if stage == "fit" or stage is None:
          # dataset for training
            self.dataset_train = ImageFolder(
                root=f"{self.data_dir}/train",
                transform=self.train_transform
            )
            # dataset for validation
            self.dataset_val = ImageFolder(
                root=f"{self.data_dir}/validation",
                transform=self.val_test_transform
            )

        if stage == "test" or stage is None:
          # dataset for testing
            self.dataset_test = ImageFolder(
                root=f"{self.data_dir}/test",
                transform=self.val_test_transform
            )

    def train_dataloader(self):
      # dataloader for train, shuffle=True to decrease chances of order memorization
        return DataLoader(self.dataset_train, batch_size=batch_size, shuffle=True)

    def val_dataloader(self):
      # dataloader for validation, data order does not change (shuffle=False), no learning in this stage
        return DataLoader(self.dataset_val, batch_size=batch_size)

    def test_dataloader(self):
      # dataloader for test, data order does not change (shuffle=False), no learning in this stage
        return DataLoader(self.dataset_test, batch_size=batch_size)


### Preventing Data Leakage

Because the dataset is split into three categories (train, validation, and test), there is a risk that data could be mixed or leaked between sets.
Therefore, each subset is loaded using ImageFolder, which ensures that data from one subset is not used in another.

### Data Augumentation On Training

I chose to apply data augmentation only on the training set because the goal is for the model to see different images at each epoch and learn general features instead of memorizing the training images. The validation and test sets must contain real and consistent images in order to perform correctly on unseen data.


### Shuffle for Training

For the training set, a DataLoader with shuffle=True is used so that images are seen in a different order at each epoch, while the validation and test sets are loaded without shuffling to ensure a fair evaluation.


## MLP Autoencoder

In [ ]:
# Simple autoencoder based on encoder + decoder
class MLPAutoencoder(pl.LightningModule):
    def __init__(self):
        super().__init__()

        input_dim = 3 * 64 * 64   # input image size ( 3 channels, 64x64)
        latent_dim = 256          # latent size

        # Encoder: compress image into latent vector
        self.encoder = nn.Sequential(
            nn.Flatten(),                 # transform image into latent vector
            nn.Linear(input_dim, 1024),
            nn.ReLU(),                    # activation function
            nn.Linear(1024, latent_dim),  # reduce dimensionality
            nn.ReLU()
        )

        # Decoder: reconstruct image from latent vector
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim),  # reconstruct to original size
            nn.Sigmoid()                 # [0,1] output
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_hat = self(x)
        # difference between input and output
        loss = F.mse_loss(x_hat, x.view(x.size(0), -1))
        self.log("ae_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
      # training optimizer
        return torch.optim.Adam(self.parameters(), lr=1e-3)


### MLP Autoencoder

I used the MLP autoencoder to improve the model's performance by recognizing relevant features that distinguish dogs from cats, rather than memorizing specific pixels.

### How it works

The encoder receives an image ( input_dim = 3 * 64 * 64 ), flattens it into a 1D vector, and then reduces it to a 256-dimensional latent vector (latent_dim = 256) that contains features of the image such as ear shape, snout shape, and fur texture. The decoder then tries to reconstruct the original image (3 * 64 * 64).

### Mean Squared Error

I used MSE to measure the difference between the reconstructed image x_hat and the original image x. To test whether the MSE loss is working properly - decreasing with each epoch - I observed how the test accuracy increases once the encoder is connected by 2%.

## CNN Model

In [ ]:
class MyCNNModel(pl.LightningModule):
    def __init__(self, num_classes=2, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.num_classes = num_classes
        self.criterion = nn.CrossEntropyLoss()

        # Lists for test results
        self.all_ground_truth = []
        self.all_predictions = []

        # Extract Features for CNN
        self.feature_extractor = nn.Sequential(
            # First layer: Input (3, 64, 64) -> Output (32, 32, 32)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second Layer: Input (32, 32, 32) -> Output (64, 16, 16)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Third Layer: Input (64, 16, 16) -> Output (128, 8, 8)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Classifier Head
        # Reduce image 3 times (2*2*2 = 8)
        # Final dimension: 64 / 8 = 8.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy(preds, y, task="multiclass", num_classes=self.num_classes)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy(preds, y, task="multiclass", num_classes=self.num_classes)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy(preds, y, task="multiclass", num_classes=self.num_classes)

        self.log("test_acc", acc, prog_bar=True)

        # Save results for interpretation
        self.all_ground_truth.append(y.cpu())
        self.all_predictions.append(preds.cpu())

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

### Training the Autoencoder

In [ ]:
# create the datamodule, loading dataset
datamodule = MyCustomDataModule(data_dir=dataset_path)
# creating autoencoder model
autoencoder = MLPAutoencoder()
# create trainer
trainer_ae = Trainer(
    max_epochs=10,                                              # loss stops decreasing
    accelerator="gpu" if torch.cuda.is_available() else "cpu",  # using gpu if it is available, else use cpu (takes more time)
    devices=1                                                   # use 1 device
)
# autoencoder train
trainer_ae.fit(autoencoder, datamodule)


### Epochs
At first, 10 epochs were chosen as a testing value. I chose 15 epochs because I observed that around this number, the ae_loss decreases sufficiently. After the first few epochs, we can notice some fluctuations in the values that do not follow a clear downward trend.

## Task 3: Define the neural network 📃


In [ ]:
class MLPWithPretrainedEncoder(LightningModule):
    def __init__(self, pretrained_encoder, num_classes=2, learning_rate=1e-3, freeze_encoder=False):
        super().__init__()
        self.num_classes = num_classes
        self.learning_rate = learning_rate

        # pretrained encoder created above
        self.encoder = pretrained_encoder

        # freeze encoder => learn to combine existent features
        # encoder used just as feature extractor, only train the classifier
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # extract features => output the class prediction
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

        # store test results
        self.all_ground_truth = []
        self.all_predictions = []

    def forward(self, x):
        z = self.encoder(x)
        logits = self.classifier(z)
        return logits

    # training: predict, compute loss & accuracy, update model
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy(preds, y, task="multiclass", num_classes=self.num_classes)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    # validation: no weight updates
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy(preds, y, task="multiclass", num_classes=self.num_classes)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    # testing: evaluate final performance on unseen data
    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy(preds, y, task="multiclass", num_classes=self.num_classes)

        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)

        # save for later analysis
        self.all_ground_truth.append(y.cpu())
        self.all_predictions.append(preds.cpu())

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


### Pretrained Encoder Vs  No Encoder Observation
### Pretrained Encoder => learns how an image is built

-> feature extractor reused by the classifier (edges, textures, shapes)

-> learns how to separate dogs from cats based on features

-> better generalization, less memorizing images

### No Encoder

-> no prior knowledge => increased learning time

-> features are learned together with classifier => more data to avoid overfitting


## Task 4: Train 📃

### Training the MLP Classification Model

In [ ]:
# created MLP Classification Model using pretrained encoder(previously trained autoencoder)
model = MLPWithPretrainedEncoder(pretrained_encoder=autoencoder.encoder, freeze_encoder=False) # use learned features during autoencoder training
# loading dataset
datamodule = MyCustomDataModule(data_dir=dataset_path)
# create trainer for classification
trainer = Trainer(
    max_epochs=20
)
# train MLP Classification Model while using autoencoder
trainer.fit(model, datamodule)


### Training the CNN Model

In [ ]:
# creating CNN Model from 0, no need to pretrain, convolutional layers directly learn features
model_cnn_direct = MyCNNModel(num_classes=2, learning_rate=1e-3)
# CNN trainer
trainer_cnn = Trainer(
    max_epochs=20,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1
)
# train CNN Model
trainer_cnn.fit(model_cnn_direct, datamodule)

## Task 5: Test ✅

### Testing MLP Classification Model

In [ ]:
trainer.test(model, datamodule=datamodule)

### Testing CNN Model

In [ ]:
trainer_cnn.test(model_cnn_direct, datamodule=datamodule)

### Results With Autoencoder
!!! Test accuracy varies between runs due to randomness in the model ( weight initialization, batch shuffling ). It is a variation of +/-2% in the data. The interpretation is based on these results:


-> train_acc = 0.56 => model learns some relevant patterns from the training data, the performance remains modest, limited fitting capacity

-> val_acc = 0.62 model generalizes reasonably well to unseen data

-> train_loss = 0.71 ~ val_loss = 0.76 stable training behavior

Difference between train and validation:

-> validation accuracy being higher than the training accuracy suggests mild underfitting

-> no evidence of excessive memorization or overfitting

Test Results:

-> test_acc = 0.615 close to val_acc = 0.62 => model generalizes consistently to unseen data

-> test_loss ≈ 0.77,comparable to training and validation loss => stable model behavior,no overfitting sign


## Task 6: Show results and experiment with splits 📃

### Ground Truth Labels Vs. Predictions MLP Model

In [ ]:
print("Ground truth length: ", len(model.all_ground_truth))
print("Predictions length: ", len(model.all_predictions))

print("Ground truth sample: ", model.all_ground_truth[0][:10])
print("Predictions sample: ", model.all_predictions[0][:10])


Comparing the first 10 values: ground truth [0,0,0,0,0,0,0,0,0,0] vs predictions [0,1,0,0,0,0,1,1,1,0], differncies observed, explaining the gap between training and validation accuracy

### Confusion Matrix for MLP Model

In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
all_true = torch.cat(model.all_ground_truth).numpy()
all_preds = torch.cat(model.all_predictions).numpy()
class_names = datamodule.dataset_test.classes
cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap=plt.cm.Purples, values_format='d')
plt.title('MLP with Pretrained Encoder')
plt.grid(False)
plt.show()

### Confusion Matrix for CNN Model

In [ ]:
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
y_true = torch.cat(model_cnn_direct.all_ground_truth).numpy()
y_pred = torch.cat(model_cnn_direct.all_predictions).numpy()
class_names = datamodule.dataset_test.classes
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('CNN Model')
plt.show()